# Tema 07: Folium

**Taller:** Análisis y Visualización Interactiva en Python
**Duración estimada de esta sesión:** 45 minutos
**Herramienta principal:** Folium (sobre Leaflet.js)
**Modalidad de práctica:** Google Colab

---

> 📌 **Nota para el profesor:** esta notebook está diseñada para proyectarse y ejecutarse en vivo. Las secciones marcadas como **Práctica guiada** se resuelven junto con el grupo; las marcadas como **Práctica independiente** las resuelven los participantes en sus propias copias de la notebook (`Archivo → Guardar una copia en Drive`).

## 🎯 Objetivos de aprendizaje

Al finalizar este tema, el participante será capaz de:

- Explicar qué es Folium y cómo se relaciona con la librería de JavaScript Leaflet.js.
- Construir un mapa interactivo con marcadores, popups y tooltips a partir de coordenadas en un DataFrame.
- Codificar variables numéricas mediante color y tamaño de marcador (`CircleMarker` + `branca.colormap`).
- Usar plugins (`MarkerCluster`, `HeatMap`) para representar grandes volúmenes de puntos geoespaciales.
- Combinar varias capas con `LayerControl` y exportar el mapa como archivo `.html` independiente.

## 🧠 Contenido teórico

### 1. ¿Qué es Folium?

**Folium** es una librería de Python que envuelve a **Leaflet.js**, una de las bibliotecas de mapas interactivos más usadas en la web (ligera, de código abierto, sin necesidad de una cuenta/API key para uso básico). Folium te permite construir mapas interactivos completos escribiendo solo Python, generando por debajo un archivo HTML con JavaScript.

Cierra el taller de forma natural: es la única herramienta enfocada específicamente en datos **geoespaciales** (coordenadas de latitud/longitud), un tipo de dato muy común (ubicación de clientes, eventos, sensores, tiendas, sismos, etc.) que ninguna de las herramientas anteriores maneja de forma nativa.

### 2. Los bloques de un mapa en Folium

- **`folium.Map(location=[lat, lon], zoom_start=...)`**: crea el mapa base, centrado en una coordenada.
- **`tiles=`**: el estilo visual del mapa base. Usaremos el valor por defecto (**OpenStreetMap**, gratuito y sin necesidad de API key). Folium admite otros proveedores (p. ej. CartoDB, Stadia Maps), pero varios de ellos ahora requieren una llave de API — para este taller nos quedamos con OpenStreetMap por simplicidad.
- **Marcadores:**
  - `folium.Marker`: pin clásico, con `popup` (aparece al hacer clic) y `tooltip` (aparece al pasar el mouse).
  - `folium.CircleMarker`: círculo cuyo **radio** y **color** puedes codificar según una variable numérica — ideal para representar magnitud, cantidad, etc.
- **Capas (`folium.FeatureGroup`, `folium.TileLayer`)** + **`folium.LayerControl()`**: permiten que el usuario active/desactive grupos de elementos o cambie el mapa base, igual que en Google Maps.

### 3. Plugins para grandes volúmenes de datos

Cuando tienes cientos o miles de puntos, mostrarlos todos como marcadores individuales satura el mapa. `folium.plugins` ofrece:

- **`MarkerCluster`**: agrupa automáticamente los marcadores cercanos en un número (círculo) que se "abre" al hacer zoom.
- **`HeatMap`**: convierte los puntos en un mapa de calor de densidad — muy útil para detectar zonas de concentración sin mostrar cada punto individual.

### 4. De Python a HTML

Un mapa de Folium (`m`) es un objeto que **se muestra automáticamente** en Colab/Jupyter (como una figura de Plotly). Para compartirlo fuera de la notebook (por ejemplo en el repositorio de GitHub del taller como página independiente), se exporta con `m.save("mapa.html")` — el resultado es un archivo interactivo que funciona en cualquier navegador, sin Python.

## ⚙️ Configuración del entorno

Cargaremos `tema07_sismos_simulados_mexico.csv`. **Importante:** este dataset es completamente **simulado** con fines educativos (coordenadas generadas aleatoriamente alrededor de zonas de referencia); las magnitudes, profundidades y fechas **no corresponden a sismos reales**.

In [ ]:
# === Carga del dataset ===
# Opción 1 (recomendada una vez publicado el repositorio del taller):
# reemplaza <usuario>/<repositorio> por la ruta real de tu repo de GitHub
# y ejecuta esta celda. Usa el botón "Raw" de GitHub para obtener la URL.
GITHUB_RAW_URL = (
    "https://raw.githubusercontent.com/<usuario>/<repositorio>/main/"
    "datasets/tema07_sismos_simulados_mexico.csv"
)

import pandas as pd

try:
    df = pd.read_csv(GITHUB_RAW_URL)
    print("Datos cargados desde GitHub ✅  ->", df.shape)
except Exception as e:
    print("No se pudo leer desde GitHub todavía (repo no configurado o sin internet).")
    print("Sube manualmente el archivo 'tema07_sismos_simulados_mexico.csv' cuando se te solicite.")
    try:
        from google.colab import files
        subido = files.upload()  # selecciona tema07_sismos_simulados_mexico.csv
        df = pd.read_csv(list(subido.keys())[0])
    except ImportError:
        # Fuera de Colab (por ejemplo, ejecución local de prueba):
        df = pd.read_csv("tema07_sismos_simulados_mexico.csv")

df.head()

## 🧭 Práctica guiada

### Paso 1 · Mapa base centrado en México

In [ ]:
import folium
from folium.plugins import MarkerCluster, HeatMap
import branca.colormap as cm

df["fecha"] = pd.to_datetime(df["fecha"])

centro_mexico = [23.6, -102.5]
m1 = folium.Map(location=centro_mexico, zoom_start=5)
m1

### Paso 2 · Marcadores individuales con popup y tooltip

Para que el mapa sea legible, tomamos solo los 25 sismos de mayor magnitud.

In [ ]:
top25 = df.sort_values("magnitud", ascending=False).head(25)

m2 = folium.Map(location=centro_mexico, zoom_start=5)

for _, sismo in top25.iterrows():
    popup_html = (
        f"<b>Región:</b> {sismo['region']}<br>"
        f"<b>Fecha:</b> {sismo['fecha'].date()}<br>"
        f"<b>Magnitud:</b> {sismo['magnitud']}<br>"
        f"<b>Profundidad:</b> {sismo['profundidad_km']} km"
    )
    folium.Marker(
        location=[sismo["latitud"], sismo["longitud"]],
        popup=folium.Popup(popup_html, max_width=250),
        tooltip=f"Magnitud {sismo['magnitud']}",
        icon=folium.Icon(color="red" if sismo["magnitud"] >= 5.5 else "orange", icon="info-sign"),
    ).add_to(m2)

m2

### Paso 3 · `CircleMarker` coloreado por magnitud (`branca.colormap`)

In [ ]:
colormap = cm.LinearColormap(
    colors=["#ffffb2", "#fd8d3c", "#bd0026"],
    vmin=df["magnitud"].min(), vmax=df["magnitud"].max(),
    caption="Magnitud simulada",
)

m3 = folium.Map(location=centro_mexico, zoom_start=5)

for _, sismo in df.iterrows():
    folium.CircleMarker(
        location=[sismo["latitud"], sismo["longitud"]],
        radius=3 + sismo["magnitud"],           # tamaño según magnitud
        color=colormap(sismo["magnitud"]),        # color según magnitud
        fill=True, fill_opacity=0.7, weight=0.5,
        popup=f"{sismo['region']} — M{sismo['magnitud']}",
    ).add_to(m3)

colormap.add_to(m3)
m3

### Paso 4 · `MarkerCluster` para todos los puntos

In [ ]:
m4 = folium.Map(location=centro_mexico, zoom_start=5)
cluster = MarkerCluster(name="Sismos simulados").add_to(m4)

for _, sismo in df.iterrows():
    folium.Marker(
        location=[sismo["latitud"], sismo["longitud"]],
        popup=f"{sismo['region']} — M{sismo['magnitud']} — {sismo['fecha'].date()}",
    ).add_to(cluster)

m4

### Paso 5 · `HeatMap` de densidad

In [ ]:
m5 = folium.Map(location=centro_mexico, zoom_start=5)

puntos_calor = df[["latitud", "longitud", "magnitud"]].values.tolist()
HeatMap(puntos_calor, radius=15, blur=20, max_zoom=8).add_to(m5)

m5

### Paso 6 · Combinar capas con `LayerControl` y exportar a HTML

In [ ]:
m6 = folium.Map(location=centro_mexico, zoom_start=5)

grupo_marcadores = folium.FeatureGroup(name="Marcadores (top 25)")
for _, sismo in top25.iterrows():
    folium.CircleMarker(
        location=[sismo["latitud"], sismo["longitud"]],
        radius=3 + sismo["magnitud"], color="crimson", fill=True, fill_opacity=0.7,
        popup=f"M{sismo['magnitud']} — {sismo['region']}",
    ).add_to(grupo_marcadores)
grupo_marcadores.add_to(m6)

grupo_calor = folium.FeatureGroup(name="Mapa de calor", show=False)
HeatMap(puntos_calor, radius=15, blur=20).add_to(grupo_calor)
grupo_calor.add_to(m6)

folium.LayerControl(collapsed=False).add_to(m6)

m6.save("mapa_sismos_simulados.html")
print("Mapa guardado como mapa_sismos_simulados.html (ábrelo en cualquier navegador).")
m6

## ✍️ Práctica independiente

**Ejercicio 1.** Filtra el DataFrame a sismos con `magnitud >= 5.0` y construye un mapa con un `CircleMarker` por cada uno (color fijo, por ejemplo `'darkred'`).

In [ ]:
# TODO: tu código aquí

**Ejercicio 2.** Construye un mapa donde el color de cada `CircleMarker` represente la `profundidad_km` en lugar de la magnitud (usa un `branca.colormap.LinearColormap` nuevo).

In [ ]:
# TODO: tu código aquí

**Ejercicio 3.** Construye un `MarkerCluster`, pero coloca en el `popup` de cada marcador la `region` en negritas y usa `folium.Icon(icon='glyphicon-warning-sign')` (o cualquier ícono de tu elección).

In [ ]:
# TODO: tu código aquí

**Ejercicio 4 (reto).** Crea un mapa con **dos** `FeatureGroup` — uno con `MarkerCluster` y otro con `HeatMap` — controlables con `LayerControl`, de forma que el usuario pueda alternar entre ambas vistas.

In [ ]:
# TODO: tu código aquí

---
### ✅ Soluciones (referencia para el profesor)

In [ ]:
# Ejercicio 1
fuertes = df[df["magnitud"] >= 5.0]
m_ej1 = folium.Map(location=centro_mexico, zoom_start=5)
for _, sismo in fuertes.iterrows():
    folium.CircleMarker(
        location=[sismo["latitud"], sismo["longitud"]],
        radius=3 + sismo["magnitud"], color="darkred", fill=True, fill_opacity=0.8,
        popup=f"M{sismo['magnitud']} — {sismo['region']}",
    ).add_to(m_ej1)
display(m_ej1)

# Ejercicio 2
colormap_prof = cm.LinearColormap(
    colors=["#deebf7", "#3182bd", "#08306b"],
    vmin=df["profundidad_km"].min(), vmax=df["profundidad_km"].max(),
    caption="Profundidad (km)",
)
m_ej2 = folium.Map(location=centro_mexico, zoom_start=5)
for _, sismo in df.iterrows():
    folium.CircleMarker(
        location=[sismo["latitud"], sismo["longitud"]],
        radius=4, color=colormap_prof(sismo["profundidad_km"]), fill=True, fill_opacity=0.7,
    ).add_to(m_ej2)
colormap_prof.add_to(m_ej2)
display(m_ej2)

# Ejercicio 3
m_ej3 = folium.Map(location=centro_mexico, zoom_start=5)
cluster_ej3 = MarkerCluster().add_to(m_ej3)
for _, sismo in df.iterrows():
    folium.Marker(
        location=[sismo["latitud"], sismo["longitud"]],
        popup=f"<b>{sismo['region']}</b><br>M{sismo['magnitud']}",
        icon=folium.Icon(icon="glyphicon-warning-sign"),
    ).add_to(cluster_ej3)
display(m_ej3)

# Ejercicio 4
m_ej4 = folium.Map(location=centro_mexico, zoom_start=5)

grupo_cluster = folium.FeatureGroup(name="Marcadores agrupados")
cluster_ej4 = MarkerCluster().add_to(grupo_cluster)
for _, sismo in df.iterrows():
    folium.Marker(location=[sismo["latitud"], sismo["longitud"]]).add_to(cluster_ej4)
grupo_cluster.add_to(m_ej4)

grupo_calor_ej4 = folium.FeatureGroup(name="Mapa de calor", show=False)
HeatMap(df[["latitud", "longitud", "magnitud"]].values.tolist()).add_to(grupo_calor_ej4)
grupo_calor_ej4.add_to(m_ej4)

folium.LayerControl(collapsed=False).add_to(m_ej4)
display(m_ej4)

## 🔎 Cierre del taller

Con Folium completamos el recorrido: partimos de **Pandas** para preparar los datos, pasamos por **Matplotlib** y **Seaborn** para visualización estática, dimos el salto a la interactividad con **Plotly**, construimos aplicaciones completas con **Dash**, exploramos series de tiempo de alto rendimiento con **Bokeh**, y cerramos con datos **geoespaciales** en Folium.

### Próximos pasos sugeridos
- Subir esta notebook y el dataset al repositorio de GitHub del taller (actualiza `GITHUB_RAW_URL` en la celda de configuración).
- Combinar herramientas: por ejemplo, un dashboard de Dash (Tema 05) que incluya un mapa de Folium embebido con `folium.Figure()._repr_html_()` dentro de un `html.Iframe`.
- Explorar `geopandas` para trabajar con polígonos (estados, municipios) además de puntos.

## 📚 Recursos adicionales

- [Documentación oficial de Folium](https://python-visualization.github.io/folium/latest/)
- [Galería de ejemplos oficiales](https://python-visualization.github.io/folium/latest/user_guide.html)
- [Plugins de Folium (MarkerCluster, HeatMap, etc.)](https://python-visualization.github.io/folium/latest/user_guide/plugins.html)
- [Documentación de Leaflet.js (la base de Folium)](https://leafletjs.com/)
- [branca.colormap — escalas de color para mapas](https://python-visualization.github.io/branca/colormap.html)